# Paper 1 — Notebook 3 of 4
## Machine-Learning Bias Correction, Leave-One-Station-Out CV & SHAP

**Run order:** NB1 → NB2 → **NB3** → NB4. Run NB1 and NB2 first.

**What NB3 does**
1. Builds the ML feature matrix: reanalysis vars + terrain + calendar + **neighbouring-station features**
2. Trains RF, XGBoost, LightGBM, Extra Trees to learn the **residual** (obs − reanalysis) — *step 1.7*
3. Evaluates every method under **Leave-One-Station-Out CV** — *step 1.8* (T7)
   — focal-filled islands (Kutubdia, Sandwip) are held out of training and scored separately
4. Rainfall detection after ML correction (adds to the T4 story)
5. **SHAP** feature importance on the best Tmax model — *step 1.9* (T8, F9 data)
6. Per-station bias/RMSE before vs after correction (T9)

**Outputs:** `T7_ml_model_comparison.csv`, `T8_shap_importance.csv`,
`T9_perstation_before_after.csv`, `loso_predictions.parquet`, `shap_values_tmax.parquet`,
`island_holdout_metrics.csv`

⚠️ **Runtime:** LOSO over 36 stations × 4 models × 3 variables is the slow part. On Kaggle CPU this is
~15–35 min. A `FAST_MODE` flag subsamples training rows for a quick smoke-test; set it False for the paper.


### Cell 1 — Imports, config, reload NB1/NB2 outputs

In [ ]:
import os, time, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width',170); pd.set_option('display.max_columns',80)

# আপনার দেওয়া এক্সাক্ট পাথগুলো এখানে বসানো হলো
OUT = '/kaggle/input/notebooks/neloypramanik4444/weatherdata-paper-v1-1/'
BASE = '/kaggle/input/datasets/neloypramanik4444/weather-dataset/dataset/C6_Terrain (slope, TPI, TWI)/'

FAST_MODE = False          # True = subsample for a quick check; False = full run for the paper
FAST_FRAC = 0.15
# Per-fold training cap. Even at full run we cap training rows per LOSO fold for tractable
# memory/time on Kaggle CPU. 120k rows (~10 stations x 12 yrs) is ample for these tree models
# and keeps results stable; raise to None if you have RAM/time to spare. Documented in Methods.
MAX_TRAIN_ROWS = 120_000
N_JOBS = 4                 # cap parallelism to bound peak memory
RANDOM_STATE = 42
FOCAL_FILLED = ['Kutubdia','Sandwip']
RAIN_THRESHOLDS=[1,10,20,50]

paired=pd.read_parquet(OUT+'paired_era5.parquet')
meta=pd.read_csv(OUT+'meta_with_cells.csv')
ter=pd.read_csv(BASE+'terrain_full.csv')
print('paired:', paired.shape)

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
try:
    from xgboost import XGBRegressor; HAS_XGB=True
except Exception: HAS_XGB=False
try:
    from lightgbm import LGBMRegressor; HAS_LGB=True
except Exception: HAS_LGB=False
print('xgboost:', HAS_XGB, '| lightgbm:', HAS_LGB)

def kge(sim,obs):
    sim=np.asarray(sim,float);obs=np.asarray(obs,float);m=np.isfinite(sim)&np.isfinite(obs)
    sim,obs=sim[m],obs[m]
    if len(obs)<3:return np.nan
    r=np.corrcoef(sim,obs)[0,1];a=sim.std()/obs.std() if obs.std() else np.nan;b=sim.mean()/obs.mean() if obs.mean() else np.nan
    return 1-np.sqrt((r-1)**2+(a-1)**2+(b-1)**2)
    
def metrics_block(sim,obs):
    sim=np.asarray(sim,float);obs=np.asarray(obs,float);m=np.isfinite(sim)&np.isfinite(obs)
    sim,obs=sim[m],obs[m]
    if len(obs)<3:return dict(n=len(obs),r=np.nan,bias=np.nan,mae=np.nan,rmse=np.nan,kge=np.nan)
    return dict(n=len(obs),r=np.corrcoef(sim,obs)[0,1],bias=(sim-obs).mean(),
                mae=np.abs(sim-obs).mean(),rmse=np.sqrt(((sim-obs)**2).mean()),kge=kge(sim,obs))
                
def contingency(sim,obs,thr):
    sim=np.asarray(sim,float);obs=np.asarray(obs,float);m=np.isfinite(sim)&np.isfinite(obs)
    sim,obs=sim[m],obs[m];fo=obs>=thr;fs=sim>=thr
    H=int((fo&fs).sum());M=int((fo&~fs).sum());F=int((~fo&fs).sum());C=int((~fo&~fs).sum());n=H+M+F+C
    pod=H/(H+M) if(H+M)else np.nan;far=F/(H+F) if(H+F)else np.nan;csi=H/(H+M+F) if(H+M+F)else np.nan
    if n: acc=(H+C)/n;rand=((H+M)*(H+F)+(C+M)*(C+F))/n**2;hss=(acc-rand)/(1-rand) if(1-rand)else np.nan
    else: hss=np.nan
    return dict(threshold=thr,H=H,M=M,F=F,C=C,POD=pod,FAR=far,CSI=csi,HSS=hss)
    
print('ready')

### Cell 2 — Neighbouring-station distance matrix
Great-circle distances between all 36 stations. For each target station and day, the *k* nearest
**other** stations supply their reanalysis anomalies as features — the residual-learning idea transferred
from the offshore-wind literature. Distances are static; the neighbour *values* are looked up per day.

In [ ]:
def haversine(lat1,lon1,lat2,lon2):
    R=6371.0
    p1,p2=np.radians(lat1),np.radians(lat2)
    dphi=np.radians(lat2-lat1); dlmb=np.radians(lon2-lon1)
    a=np.sin(dphi/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

coords=meta.set_index('station_name')[['lat','lon']]
stations=list(coords.index)
D=pd.DataFrame(index=stations, columns=stations, dtype=float)
for a in stations:
    for b in stations:
        D.loc[a,b]=0.0 if a==b else haversine(coords.loc[a,'lat'],coords.loc[a,'lon'],
                                               coords.loc[b,'lat'],coords.loc[b,'lon'])
K_NEIGH=3
neighbours={s:[x for x in D.loc[s].sort_values().index if x!=s][:K_NEIGH] for s in stations}
print('Example neighbours:')
for s in ['Dhaka','Hatiya','Sylhet']:
    print(' ', s, '->', neighbours[s], '(km:', [round(D.loc[s,n],0) for n in neighbours[s]], ')')

### Cell 3 — Assemble the ML feature matrix
Features per station-day:
* reanalysis: `t2m_max, t2m_min, era5_prcp, ssrd, ws2, rh_mean, vpd, swvl1, sp`
* terrain: `land_frac_9km, elev_std_9km, roughness_5km, dist_water_km, elev_dem, northness, eastness`
* calendar: day-of-year sin/cos, month
* neighbour features: mean of the k nearest stations' same-day reanalysis value + their mean anomaly

Target = **residual** (obs − reanalysis) for each of Tmax, Tmin, precip. Learning the residual keeps the
physical signal in the reanalysis and lets the model focus on the correctable bias.

In [ ]:
df=paired.copy()
df['doy']=df['date'].dt.dayofyear
df['month']=df['date'].dt.month
df['doy_sin']=np.sin(2*np.pi*df['doy']/365.25)
df['doy_cos']=np.cos(2*np.pi*df['doy']/365.25)

tcols=['land_frac_9km','elev_std_9km','roughness_5km','dist_water_km',
       'elev_dem','northness','eastness','tpi_5km']
df=df.merge(ter[['station_name']+tcols], on='station_name', how='left')

# neighbour same-day reanalysis lookup — VECTORISED (build date x station table, melt, merge)
def add_neighbour_feats(df, value_col):
    piv=df.pivot_table(index='date', columns='station_name', values=value_col, aggfunc='mean')
    nb_tbl=pd.DataFrame({s: piv[neighbours[s]].mean(axis=1) for s in stations})  # date x target-station
    long=nb_tbl.reset_index().melt(id_vars='date', var_name='station_name', value_name='nbv')
    return df.merge(long, on=['date','station_name'], how='left')['nbv'].values

df['nb_tmax']=add_neighbour_feats(df,'t2m_max')
df['nb_tmin']=add_neighbour_feats(df,'t2m_min')
df['nb_prcp']=add_neighbour_feats(df,'era5_prcp')

BASE_FEATS=['t2m_max','t2m_min','era5_prcp','ssrd','ws2','rh_mean','vpd','swvl1','sp']
CAL_FEATS =['doy_sin','doy_cos','month']
NB_FEATS  =['nb_tmax','nb_tmin','nb_prcp']
FEATURES = BASE_FEATS + tcols + CAL_FEATS + NB_FEATS
FEATURES_NO_NB = BASE_FEATS + tcols + CAL_FEATS   # for the single-station ablation

# residual targets
df['res_tmax']=df['obs_tmax']-df['t2m_max']
df['res_tmin']=df['obs_tmin']-df['t2m_min']
df['res_prcp']=df['obs_prcp']-df['era5_prcp']

print('feature count (with neighbours):', len(FEATURES))
print('rows with full features:', df[FEATURES].notna().all(axis=1).sum())

### Cell 4 — Model zoo and Leave-One-Station-Out engine
Each fold holds out one station entirely (spatial generalisation, no autocorrelation leakage). Focal-filled
island stations are **never** in the training pool; they are scored as an independent island test. For each
held-out station we predict its residual, add it back to the reanalysis, and score the corrected series.

In [ ]:
# ============================================================
#  GPU-aware models + faster LOSO engine  (science unchanged)
# ============================================================
# GPU thakle XGB/LGBM auto GPU chalabe; na thakle nijei CPU-te fall back korbe (crash na).
# RF/ExtraTrees sklearn-e GPU pare na, tai oder n_jobs=-1 (sob core) + max_samples diye fast kora holo.

import numpy as np, pandas as pd, time
from sklearn.base import clone

# --- GPU auto-detect (shudhu XGB/LGBM-er jonno) ---
def _gpu_available():
    try:
        import subprocess
        subprocess.check_output(['nvidia-smi'], stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False
USE_GPU = _gpu_available()
print('GPU detected:', USE_GPU)

def make_models():
    m = {
        # RF/ExtraTrees: CPU-only, kintu sob core + per-tree subsample => onek fast, result ~same
        'RF': RandomForestRegressor(
            n_estimators=150, max_depth=20, min_samples_leaf=5,
            max_samples=0.5, bootstrap=True,           # <-- per-tree te half data => 2x+ fast
            n_jobs=-1, random_state=RANDOM_STATE),
        'ExtraTrees': ExtraTreesRegressor(
            n_estimators=200, max_depth=25, min_samples_leaf=5,
            max_samples=0.5, bootstrap=True,           # <-- same trick
            n_jobs=-1, random_state=RANDOM_STATE),
    }
    if HAS_XGB:
        xgb_kw = dict(n_estimators=400, max_depth=6, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8,
                      random_state=RANDOM_STATE, tree_method='hist', n_jobs=-1)
        if USE_GPU:
            xgb_kw['device'] = 'cuda'                  # GPU thakle eta chalu hobe
        m['XGBoost'] = XGBRegressor(**xgb_kw)
    if HAS_LGB:
        lgb_kw = dict(n_estimators=500, num_leaves=31, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8,
                      random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)
        if USE_GPU:
            lgb_kw['device'] = 'gpu'                   # GPU thakle eta chalu hobe
        m['LightGBM'] = LGBMRegressor(**lgb_kw)
    return m

TRAIN_POOL = [s for s in stations if s not in FOCAL_FILLED]

def loso_run(df, target_res, mod_col, obs_col, feats, label, models_subset=None):
    rows = []
    models = make_models()
    if models_subset:
        models = {k: v for k, v in models.items() if k in models_subset}

    # --- ekbar-i dropna kore per-station numpy array ready kore rakhi (bar-baর masking bondho) ---
    need = feats + [target_res]
    base = df[df.station_name.isin(TRAIN_POOL)].dropna(subset=need)
    Xall = base[feats].to_numpy(np.float32)                 # float32 => kom RAM, fast
    yall = base[target_res].to_numpy(np.float32)
    sall = base['station_name'].to_numpy()
    # test-side (feats-e NaN nai emon row)
    test_ok = df.dropna(subset=feats)

    for mname, proto in models.items():
        preds = pd.Series(index=df.index, dtype=float)
        for held in TRAIN_POOL:
            m_tr = sall != held                              # ei station bade baki sob
            Xtr, ytr = Xall[m_tr], yall[m_tr]
            if FAST_MODE and len(Xtr) > 0:
                k = int(len(Xtr) * FAST_FRAC)
                idx = np.random.RandomState(RANDOM_STATE).choice(len(Xtr), k, replace=False)
                Xtr, ytr = Xtr[idx], ytr[idx]
            elif MAX_TRAIN_ROWS and len(Xtr) > MAX_TRAIN_ROWS:
                idx = np.random.RandomState(RANDOM_STATE).choice(len(Xtr), MAX_TRAIN_ROWS, replace=False)
                Xtr, ytr = Xtr[idx], ytr[idx]

            te = test_ok[test_ok.station_name == held]
            if len(Xtr) < 200 or len(te) == 0:
                continue
            mdl = clone(proto)
            mdl.fit(Xtr, ytr)
            resid_hat = mdl.predict(te[feats].to_numpy(np.float32))
            preds.loc[te.index] = te[mod_col].to_numpy() + resid_hat

        te_all = df.loc[preds.notna()]
        mb = metrics_block(preds.loc[te_all.index].values, te_all[obs_col].values)
        mb.update(variable=label, method=mname, cv='LOSO',
                  features=('with_nb' if 'nb_tmax' in feats else 'no_nb'))
        rows.append(mb)
        df[f'pred_{label}_{mname}'] = preds        # NB4 ei column-gulো use kore
    return pd.DataFrame(rows)

t0 = time.time()
res_tmax = loso_run(df, 'res_tmax', 't2m_max', 'obs_tmax', FEATURES, 'Tmax')
print('Tmax LOSO done in %.0fs' % (time.time() - t0))
res_tmax.round(3)

### Cell 5 — LOSO for Tmin and precipitation, plus the neighbour-feature ablation (RQ1.3)

In [ ]:
import os, time
OUT = '/kaggle/working/' if os.path.exists('/kaggle/working/') else './'

t0 = time.time()
res_tmin = loso_run(df, 'res_tmin', 't2m_min', 'obs_tmin', FEATURES, 'Tmin')
res_prcp = loso_run(df, 'res_prcp', 'era5_prcp', 'obs_prcp', FEATURES, 'Precip')
print('Tmin+Precip LOSO in %.0fs' % (time.time() - t0))

# Ablation: neighbour-assisted vs single-station (ekta model-i, tai fast)
abl_model = 'ExtraTrees'
t0 = time.time()
res_tmax_nonb = loso_run(df, 'res_tmax', 't2m_max', 'obs_tmax', FEATURES_NO_NB, 'Tmax',
                         models_subset=[abl_model])
print('ablation in %.0fs' % (time.time() - t0))

T7 = pd.concat([res_tmax, res_tmin, res_prcp, res_tmax_nonb], ignore_index=True)
raw_rows = []
for lab, mod, obs in [('Tmax', 't2m_max', 'obs_tmax'),
                      ('Tmin', 't2m_min', 'obs_tmin'),
                      ('Precip', 'era5_prcp', 'obs_prcp')]:
    sub = df[df.station_name.isin(TRAIN_POOL)]
    mb = metrics_block(sub[mod].values, sub[obs].values)
    mb.update(variable=lab, method='RAW (ERA5)', cv='-', features='-')
    raw_rows.append(mb)
T7 = pd.concat([pd.DataFrame(raw_rows), T7], ignore_index=True)
for c in ['r', 'bias', 'mae', 'rmse', 'kge']:
    T7[c] = T7[c].round(3)
T7 = T7[['variable', 'method', 'cv', 'features', 'n', 'r', 'bias', 'mae', 'rmse', 'kge']]

T7.to_csv(OUT + 'T7_ml_model_comparison.csv', index=False)
print('=== TABLE T7 — ML model comparison under LOSO ===')
print(T7.to_string(index=False))

nb_kge = T7.query("variable=='Tmax' and method==@abl_model and features=='with_nb'")['kge'].values
no_kge = T7.query("variable=='Tmax' and method==@abl_model and features=='no_nb'")['kge'].values
if len(nb_kge) and len(no_kge):
    print(f'\nRQ1.3 neighbour-assisted vs single-station ({abl_model}, Tmax): '
          f'KGE {no_kge[0]:.3f} -> {nb_kge[0]:.3f} (delta {nb_kge[0]-no_kge[0]:+.3f})')

### Cell 6 — Island focal-fill holdout test (Kutubdia, Sandwip)
Train the best Tmax model on all mainland stations, predict the two focal-filled islands. This tests how correction behaves at sites that were never native ERA5-Land.

In [ ]:
import os
from sklearn.base import clone

# ফাইল সেভ করার জন্য Kaggle-এর সঠিক ডিরেক্টরি (working folder) সেট করে দেওয়া হলো
OUT = '/kaggle/working/' if os.path.exists('/kaggle/working/') else './'

best_row=T7.query("variable=='Tmax' and cv=='LOSO' and features=='with_nb'").sort_values('kge',ascending=False).iloc[0]
best_model_name=best_row['method']
print('Best Tmax LOSO model:', best_model_name)

proto=make_models()[best_model_name]
tr=df[df.station_name.isin(TRAIN_POOL)].dropna(subset=FEATURES+['res_tmax'])
mdl=clone(proto).fit(tr[FEATURES].values, tr['res_tmax'].values)

isl_rows=[]
for s in FOCAL_FILLED:
    te=df[df.station_name==s].dropna(subset=FEATURES)
    if len(te)==0: 
        isl_rows.append(dict(station=s, note='no rows')); continue
    corr=te['t2m_max'].values + mdl.predict(te[FEATURES].values)
    raw_mb=metrics_block(te['t2m_max'].values, te['obs_tmax'].values)
    cor_mb=metrics_block(corr, te['obs_tmax'].values)
    isl_rows.append(dict(station=s, n=raw_mb['n'],
                         raw_bias=round(raw_mb['bias'],3), corr_bias=round(cor_mb['bias'],3),
                         raw_rmse=round(raw_mb['rmse'],3), corr_rmse=round(cor_mb['rmse'],3),
                         raw_kge=round(raw_mb['kge'],3), corr_kge=round(cor_mb['kge'],3)))
island=pd.DataFrame(isl_rows)

# এখন ফাইলটি সফলভাবে সেভ হবে
island.to_csv(OUT+'island_holdout_metrics.csv', index=False)

print('=== Island focal-fill holdout (Tmax) ===')
print(island.to_string(index=False))

### Cell 7 — SHAP interpretation of the best Tmax model (T8, F9 data) — *step 1.9*

In [ ]:
try:
    import shap; HAS_SHAP=True
except Exception:
    HAS_SHAP=False
    print('shap not available; computing tree impurity importance as fallback')

# fit once on a manageable sample for SHAP speed
samp=tr.sample(n=min(40000,len(tr)), random_state=RANDOM_STATE)
mdl_shap=clone(proto).fit(samp[FEATURES].values, samp['res_tmax'].values)

if HAS_SHAP:
    expl=shap.TreeExplainer(mdl_shap)
    Xs=samp[FEATURES].sample(n=min(8000,len(samp)), random_state=RANDOM_STATE)
    sv=expl.shap_values(Xs.values)
    mean_abs=np.abs(sv).mean(axis=0)
    T8=pd.DataFrame({'feature':FEATURES,'mean_abs_shap':mean_abs}).sort_values('mean_abs_shap',ascending=False)
    # persist raw shap for the beeswarm in NB4
    shp=pd.DataFrame(sv, columns=FEATURES); shp_x=Xs.reset_index(drop=True)
    shp.to_parquet(OUT+'shap_values_tmax.parquet', index=False)
    shp_x.to_parquet(OUT+'shap_features_tmax.parquet', index=False)
else:
    imp=mdl_shap.feature_importances_
    T8=pd.DataFrame({'feature':FEATURES,'mean_abs_shap':imp}).sort_values('mean_abs_shap',ascending=False)
T8['mean_abs_shap']=T8['mean_abs_shap'].round(4)
T8.to_csv(OUT+'T8_shap_importance.csv', index=False)
print('=== TABLE T8 — SHAP feature importance (Tmax residual model) ===')
print(T8.to_string(index=False))

### Cell 8 — Per-station before/after (T9) and detection after ML correction

In [ ]:
# use the best model's LOSO predictions column
pred_col=f'pred_Tmax_{best_model_name}'
rows=[]
for s,g in df.groupby('station_name'):
    raw=metrics_block(g['t2m_max'].values, g['obs_tmax'].values)
    if pred_col in g and g[pred_col].notna().any():
        cor=metrics_block(g[pred_col].values, g['obs_tmax'].values)
    else:
        cor=dict(bias=np.nan,rmse=np.nan,kge=np.nan)
    rows.append(dict(station_name=s,
                     raw_bias=round(raw['bias'],3), corr_bias=round(cor['bias'],3),
                     raw_rmse=round(raw['rmse'],3), corr_rmse=round(cor['rmse'],3),
                     raw_kge=round(raw['kge'],3), corr_kge=round(cor.get('kge',np.nan),3) if cor.get('kge')==cor.get('kge') else np.nan,
                     focal_filled=s in FOCAL_FILLED))
T9=pd.DataFrame(rows)
T9.to_csv(OUT+'T9_perstation_before_after.csv', index=False)
print('=== TABLE T9 — per-station Tmax before/after (head) ===')
print(T9.head(10).to_string(index=False))
print('\nNetwork mean |bias|: raw %.3f -> corrected %.3f'%(
    T9['raw_bias'].abs().mean(), T9['corr_bias'].abs().mean()))

# detection after ML precip correction
pred_p=f'pred_Precip_{best_model_name}' if f'pred_Precip_{best_model_name}' in df else None
if pred_p is None:
    # pick any available precip prediction
    cand=[c for c in df.columns if c.startswith('pred_Precip_')]
    pred_p=cand[0] if cand else None
det=[]
if pred_p:
    sub=df.loc[df[pred_p].notna()]
    for thr in RAIN_THRESHOLDS:
        c=contingency(sub['era5_prcp'].values, sub['obs_prcp'].values, thr); c.update(stage='raw')
        det.append(c)
        c=contingency(sub[pred_p].values, sub['obs_prcp'].values, thr); c.update(stage=f'ML ({pred_p.split("_")[-1]})')
        det.append(c)
    detdf=pd.DataFrame(det)[['stage','threshold','POD','FAR','CSI','HSS']].round(3)
    detdf.to_csv(OUT+'detection_after_ml.csv', index=False)
    print('\n=== Detection after ML precip correction ===')
    print(detdf.to_string(index=False))

### Cell 9 — Persist LOSO predictions for NB4 reconstruction & figures

In [ ]:
pred_cols=[c for c in df.columns if c.startswith('pred_')]
keep=df[['station_name','date','obs_tmax','obs_tmin','obs_prcp',
         't2m_max','t2m_min','era5_prcp']+pred_cols].copy()
keep.to_parquet(OUT+'loso_predictions.parquet', index=False)
print('Wrote loso_predictions.parquet with prediction columns:')
print(pred_cols)
print('\nBest models chosen — Tmax:', best_model_name)
print('NB3 complete. Next: run Notebook 4 (reconstruction, all figures, Zenodo export).')

In [ ]:
import os, shutil, glob
from IPython.display import FileLink

OUT = '/kaggle/working/' if os.path.exists('/kaggle/working/') else './'
zip_base = os.path.join(OUT, 'paper1_all_outputs')
stage = os.path.join(OUT, '_paper1_zip_stage')

# clean stage
if os.path.exists(stage):
    shutil.rmtree(stage)
os.makedirs(stage, exist_ok=True)

# collect everything we produced (csv, parquet, json, md, png) + figures folder
patterns = ['*.csv', '*.parquet', '*.json', '*.md', '*.png']
copied = 0
for pat in patterns:
    for f in glob.glob(os.path.join(OUT, pat)):
        shutil.copy(f, stage); copied += 1

fig_dir = os.path.join(OUT, 'figures')
if os.path.isdir(fig_dir):
    shutil.copytree(fig_dir, os.path.join(stage, 'figures'), dirs_exist_ok=True)
    copied += len(glob.glob(os.path.join(fig_dir, '*')))

# make the zip
zip_path = shutil.make_archive(zip_base, 'zip', stage)
shutil.rmtree(stage)

size_mb = os.path.getsize(zip_path) / 1e6
print(f'Zipped {copied} files -> {zip_path} ({size_mb:.1f} MB)')
print('Download link (also in the Output panel on the right):')
FileLink(zip_path)